# IOAI — 2026 Contest Star Galaxy Quasar (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.listdir('data'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-contest-star-galaxy-quasar/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Star/Galaxy/Quasar — 모범답안 (색지수 + 균형 랜덤포레스트)

천체 분류의 핵심은 **색지수**(밴드 차이 u−g, g−r, r−i, i−z)다. 5개 광도에 색지수를 더하고, 불균형·recall
강조를 위해 **class_weight='balanced'** 로 랜덤포레스트를 학습한다. F2 ≈ 0.84 (다수결 대비 큰 향상,
과학위원회 참고점수 0.82).

In [ ]:
import pandas as pd
train = pd.read_csv("data/train.csv")            # objid, ra, dec, modelMag_*, type
test  = pd.read_csv("data/testdata.csv")         # (type 없음)
MAGS = ["modelMag_u","modelMag_g","modelMag_r","modelMag_i","modelMag_z"]
print(train.shape, "| classes:", train.type.value_counts().to_dict())

In [ ]:
from sklearn.ensemble import RandomForestClassifier
MAGS = ["modelMag_u","modelMag_g","modelMag_r","modelMag_i","modelMag_z"]
def features(df):
    x = df[MAGS].copy()
    for a, b in [("modelMag_u","modelMag_g"),("modelMag_g","modelMag_r"),
                 ("modelMag_r","modelMag_i"),("modelMag_i","modelMag_z")]:
        x[f"{a}-{b}"] = df[a] - df[b]
    return x
clf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=0, n_jobs=-1)
clf.fit(features(train), train["type"])
pred = clf.predict(features(test))
pd.DataFrame({"objid": test["objid"], "type": pred}).to_csv("submission.csv", index=False)
print("saved submission.csv", len(pred))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)